# Query a Hash-Sorted STAC GeoParquet File

This tutorial shows how to query a hash-sorted STAC GeoParquet file with DuckDB.

The main idea is simple: users still query normal STAC fields like collection, datetime, bounding box, cloud cover, and geometry. The hash helps because the file was sorted by a compact space-time key before it was written.


## Setup

Create a small project for the tutorial with `uv`:

```sh
uv init stac-hash-query
cd stac-hash-query
uv add stac-hash duckdb pyarrow ipykernel
uv run python -m ipykernel install --user --name stac-hash-query
```

Then open this notebook using the `stac-hash-query` kernel.

If you already ran the create tutorial, use the `example-scenes-2026.parquet` file from that tutorial. This notebook can also create a tiny fallback file so the queries run end to end.


## Open The Dataset

Start by pointing DuckDB at the GeoParquet file. For a local tutorial run, this is just a file path. For a published dataset, this could be an HTTPS URL, S3 path, or glob of many Parquet files.


In [ ]:
from pathlib import Path

PARQUET_PATH = Path("example-scenes-2026.parquet")
PARQUET_PATH


## Create A Small Fallback File

Skip this section if you already have `example-scenes-2026.parquet` from the create tutorial. The fallback keeps this notebook runnable on its own by writing a small hash-sorted table with realistic STAC-style fields.


In [ ]:
from datetime import UTC, datetime

import pyarrow as pa
import pyarrow.parquet as pq
from stac_hash import Hasher


def parse_datetime(value: str) -> datetime:
    return datetime.fromisoformat(value.replace("Z", "+00:00")).astimezone(UTC)


def bbox_center(bbox: list[float]) -> tuple[float, float]:
    west, south, east, north = bbox
    return ((west + east) / 2, (south + north) / 2)


if not PARQUET_PATH.exists():
    spatial_extent = [-110.0, 37.0, -102.0, 42.0]
    temporal_start = datetime(2026, 1, 1, tzinfo=UTC)
    temporal_end = datetime(2027, 1, 1, tzinfo=UTC)
    hasher = Hasher(temporal_start, temporal_end, bbox=tuple(spatial_extent))

    source_items = [
        ("item-001", "example-scenes", "2026-01-15T18:00:00Z", [-105.2, 39.8, -105.0, 40.0], 8.0),
        ("item-002", "example-scenes", "2026-01-16T18:00:00Z", [-104.9, 40.0, -104.7, 40.2], 12.0),
        ("item-003", "example-scenes", "2026-02-03T18:00:00Z", [-106.1, 38.9, -105.9, 39.1], 3.0),
        ("item-004", "example-scenes", "2026-03-22T18:00:00Z", [-103.4, 40.9, -103.2, 41.1], 33.0),
        ("item-005", "example-scenes", "2026-01-18T18:00:00Z", [-108.5, 37.4, -108.3, 37.6], 5.0),
        ("item-006", "other-scenes", "2026-01-19T18:00:00Z", [-105.1, 39.9, -104.9, 40.1], 2.0),
    ]

    rows = []
    for item_id, collection, datetime_text, bbox, cloud_cover in source_items:
        longitude, latitude = bbox_center(bbox)
        item_datetime = parse_datetime(datetime_text)
        rows.append(
            {
                "id": item_id,
                "collection": collection,
                "datetime": item_datetime,
                "bbox_xmin": bbox[0],
                "bbox_ymin": bbox[1],
                "bbox_xmax": bbox[2],
                "bbox_ymax": bbox[3],
                "eo:cloud_cover": cloud_cover,
                "hash:hash": hasher.hash_encoded(item_datetime, longitude, latitude, "base16"),
            }
        )

    table = pa.Table.from_pylist(rows).sort_by([("hash:hash", "ascending")])
    pq.write_table(table, PARQUET_PATH, row_group_size=2)

PARQUET_PATH.exists()


The fallback writes very small row groups so it is easier to inspect the file. Real datasets should use row groups sized for efficient reads and useful statistics.


## Inspect The Table

DuckDB can query Parquet files directly. Start by checking the schema and a few rows ordered by `hash:hash`.


In [ ]:
import duckdb

con = duckdb.connect()
parquet_path = PARQUET_PATH.as_posix()

con.sql(
    f"""
    DESCRIBE SELECT *
    FROM read_parquet('{parquet_path}')
    """
).show()


In [ ]:
con.sql(
    f"""
    SELECT id, collection, datetime, "eo:cloud_cover", "hash:hash"
    FROM read_parquet('{parquet_path}')
    ORDER BY "hash:hash"
    """
).show()


## Run A STAC-Style Search

The query still uses normal STAC-style predicates. The hash is useful because it influenced the physical layout of the file before the query started.


In [ ]:
search = con.sql(
    f"""
    SELECT id, datetime, "eo:cloud_cover", "hash:hash"
    FROM read_parquet('{parquet_path}')
    WHERE collection = 'example-scenes'
      AND datetime >= TIMESTAMPTZ '2026-01-01T00:00:00Z'
      AND datetime < TIMESTAMPTZ '2026-02-01T00:00:00Z'
      AND bbox_xmax >= -105.3
      AND bbox_xmin <= -104.6
      AND bbox_ymax >= 39.7
      AND bbox_ymin <= 40.3
    ORDER BY datetime
    """
)

search.show()


This is the same kind of filter a STAC API might apply: collection, time range, and bounding box. Nothing in the SQL requires a custom index service.


## Page Through Results

For search result pages, you can choose the ordering that fits the product. Ordering by datetime is familiar. Ordering by `hash:hash` groups each page by space and time, which can be useful for map browsing or downstream batch reads.


In [ ]:
con.sql(
    f"""
    SELECT id, datetime, "hash:hash"
    FROM read_parquet('{parquet_path}')
    WHERE collection = 'example-scenes'
    ORDER BY "hash:hash"
    LIMIT 3
    """
).show()


## Add Attribute Filters

Hash ordering does not stop you from using ordinary metadata filters. For example, filter by cloud cover after applying collection, time, and bbox predicates.


In [ ]:
con.sql(
    f"""
    SELECT id, datetime, "eo:cloud_cover", "hash:hash"
    FROM read_parquet('{parquet_path}')
    WHERE collection = 'example-scenes'
      AND datetime >= TIMESTAMPTZ '2026-01-01T00:00:00Z'
      AND datetime < TIMESTAMPTZ '2026-03-01T00:00:00Z'
      AND bbox_xmax >= -106.5
      AND bbox_xmin <= -104.0
      AND bbox_ymax >= 38.5
      AND bbox_ymin <= 40.5
      AND "eo:cloud_cover" < 10
    ORDER BY "eo:cloud_cover", datetime
    """
).show()


## Inspect The Query Plan

Use `EXPLAIN ANALYZE` when you want to understand whether a layout is helping. On a tiny tutorial file, the timings will not be meaningful. On a real dataset, look for how many files or row groups DuckDB scans and how much time is spent in `READ_PARQUET`.


In [ ]:
plan = con.sql(
    f"""
    EXPLAIN ANALYZE
    SELECT count(*)
    FROM read_parquet('{parquet_path}')
    WHERE collection = 'example-scenes'
      AND datetime >= TIMESTAMPTZ '2026-01-01T00:00:00Z'
      AND datetime < TIMESTAMPTZ '2026-02-01T00:00:00Z'
      AND bbox_xmax >= -105.3
      AND bbox_xmin <= -104.6
      AND bbox_ymax >= 39.7
      AND bbox_ymin <= 40.3
    """
).fetchall()

print(plan[0][1])


The important thing to compare is not a single timing from a tiny local file. Compare the same query against different physical layouts: unsorted, datetime-sorted, and hash-sorted. The hash-sorted layout is useful when it lets the engine skip more irrelevant Parquet data for common spatial-temporal searches.


## Query Remote Data

For object storage, the SQL shape is the same. DuckDB's `httpfs` extension can read public S3 or HTTPS Parquet files directly.

```sql
INSTALL httpfs;
LOAD httpfs;

SELECT count(*)
FROM read_parquet('s3://example-bucket/stac/*.parquet')
WHERE collection = 'sentinel-2-l2a'
  AND datetime >= TIMESTAMPTZ '2026-01-01T00:00:00Z'
  AND datetime < TIMESTAMPTZ '2026-02-01T00:00:00Z'
  AND bbox_xmax >= -105.3
  AND bbox_xmin <= -104.6
  AND bbox_ymax >= 39.7
  AND bbox_ymin <= 40.3;
```

This is where layout usually matters most. On object storage, skipping files and row groups can avoid network requests and range reads, not just CPU work.


## What To Remember

- Keep querying normal STAC fields.
- Use `hash:hash` as a physical ordering key, not as a replacement for filters.
- Use `EXPLAIN ANALYZE` to compare layouts on real data.
- Expect the strongest gains when spatial-temporal filters can skip files or row groups.

The hash pays off when it is used before query time, while writing the GeoParquet layout.
